# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster KMeans"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
30,2022-09-02 06:00:00,0.000000,17,7,91,0,4,15,6,Nublado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,7,94,0,3,16,7,Nublado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,5,97,0,3,15,8,Nublado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,0,93,1,2,16,9,Nublado,Lluvioso,438.814997,7720.582326
34,2022-09-02 10:00:00,5030.740421,18,0,85,2,2,15,10,Nublado,Lluvioso,5908.000884,9433.109309
39,2022-09-02 15:00:00,24647.568577,26,7,33,5,4,8,15,Nublado,Lluvioso,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,7,34,4,4,9,16,Nublado,Lluvioso,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,7,36,2,4,12,17,Nublado,Lluvioso,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,7,39,1,4,12,18,Nublado,Lluvioso,24281.956494,29900.303971
43,2022-09-02 19:00:00,11972.590689,25,7,44,1,4,12,19,Nublado,Lluvioso,22733.515002,18282.505369


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,17,7,91,0,4,15,6,0.000000,0.000000
31,17,7,94,0,3,16,7,0.000000,6.584959
32,16,5,97,0,3,15,8,0.000000,560.422022
33,17,0,93,1,2,16,9,438.814997,7720.582326
34,18,0,85,2,2,15,10,5908.000884,9433.109309
...,...,...,...,...,...,...,...,...,...
18273,14,0,87,1,4,11,8,67.000000,7302.000000
18274,15,0,83,2,4,12,9,7356.000000,18014.000000
18275,17,0,71,4,3,12,10,17638.000000,23010.000000
18276,19,0,60,5,3,11,11,23339.000000,26156.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
30,0.000000
31,0.000000
32,438.814997
33,5908.000884
34,5030.740421
...,...
18273,7356.000000
18274,17638.000000
18275,23339.000000
18276,26323.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 3649, y_train: 3649
X_val: 782, y_val: 782
X_test: 783, y_test: 783


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[5.00000000e-01 7.77777778e-02 9.04255319e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [5.00000000e-01 7.77777778e-02 9.36170213e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.68750000e-01 5.55555556e-02 9.68085106e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [2.50000000e-01 1.11111111e-02 9.78723404e-01 ... 6.66666667e-02
  0.00000000e+00 0.00000000e+00]
 [2.18750000e-01 0.00000000e+00 1.00000000e+00 ... 1.33333333e-01
  0.00000000e+00 5.96666667e-03]
 [1.87500000e-01 0.00000000e+00 1.00000000e+00 ... 2.00000000e-01
  9.83333333e-03 3.62200000e-01]]
(3649, 9)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.50000,0.077778,0.904255,0.000000,0.666667,0.789474,0.000000,0.000000,0.000000
31,0.50000,0.077778,0.936170,0.000000,0.333333,0.842105,0.066667,0.000000,0.000219
32,0.46875,0.055556,0.968085,0.000000,0.333333,0.789474,0.133333,0.000000,0.018681
33,0.50000,0.000000,0.925532,0.083333,0.000000,0.842105,0.200000,0.014627,0.257353
34,0.53125,0.000000,0.840426,0.166667,0.000000,0.789474,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
12958,0.53125,0.000000,0.446809,0.000000,0.333333,0.368421,1.000000,0.000000,0.000000
12967,0.28125,0.011111,0.978723,0.000000,1.000000,0.473684,0.000000,0.000000,0.000000
12968,0.25000,0.011111,0.978723,0.000000,1.000000,0.473684,0.066667,0.000000,0.000000
12969,0.21875,0.000000,1.000000,0.000000,0.333333,0.421053,0.133333,0.000000,0.005967


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.28125    0.         0.82978723 ... 0.26666667 0.31593333 0.63086667]
 [0.375      0.         0.67021277 ... 0.33333333 0.65066667 0.66426667]
 [0.46875    0.         0.54255319 ... 0.4        0.67886667 0.6496    ]
 ...
 [0.75       0.02222222 0.43617021 ... 0.86666667 0.34693333 0.1592    ]
 [0.6875     0.06666667 0.5106383  ... 0.93333333 0.20373333 0.0115    ]
 [0.46875    0.07777778 0.87234043 ... 0.         0.         0.        ]]
(782, 9)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
12971,0.28125,0.000000,0.829787,0.166667,0.333333,0.368421,0.266667,0.315933,0.630867
12972,0.37500,0.000000,0.670213,0.166667,0.333333,0.368421,0.333333,0.650667,0.664267
12973,0.46875,0.000000,0.542553,0.250000,0.333333,0.368421,0.400000,0.678867,0.649600
12974,0.53125,0.000000,0.457447,0.250000,0.333333,0.368421,0.466667,0.656667,0.798233
12981,0.53125,0.000000,0.436170,0.000000,0.666667,0.368421,0.933333,0.042467,0.000000
...,...,...,...,...,...,...,...,...,...
16383,0.78125,0.000000,0.382979,0.750000,0.666667,0.631579,0.533333,0.900000,0.362167
16387,0.78125,0.000000,0.382979,0.166667,0.666667,0.631579,0.800000,0.379967,0.348467
16388,0.75000,0.022222,0.436170,0.000000,1.000000,0.684211,0.866667,0.346933,0.159200
16389,0.68750,0.066667,0.510638,0.000000,0.333333,0.684211,0.933333,0.203733,0.011500


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.46875    0.06666667 0.87234043 ... 0.06666667 0.         0.06736667]
 [0.46875    0.02222222 0.86170213 ... 0.13333333 0.0214     0.4988    ]
 [0.5        0.02222222 0.80851064 ... 0.2        0.2268     0.8451    ]
 ...
 [0.5        0.         0.69148936 ... 0.26666667 0.58793333 0.767     ]
 [0.5625     0.         0.57446809 ... 0.33333333 0.77796667 0.87186667]
 [0.6875     0.         0.40425532 ... 0.46666667 0.8759     0.8551    ]]
(783, 9)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
16400,0.46875,0.066667,0.872340,0.000000,1.000000,0.736842,0.066667,0.000000,0.067367
16401,0.46875,0.022222,0.861702,0.083333,1.000000,0.736842,0.133333,0.021400,0.498800
16402,0.50000,0.022222,0.808511,0.166667,1.000000,0.736842,0.200000,0.226800,0.845100
16403,0.59375,0.022222,0.702128,0.166667,1.000000,0.736842,0.266667,0.350033,0.895933
16404,0.65625,0.022222,0.606383,0.583333,0.333333,0.736842,0.333333,0.438233,0.900000
...,...,...,...,...,...,...,...,...,...
18273,0.40625,0.000000,0.861702,0.083333,0.666667,0.578947,0.133333,0.002233,0.243400
18274,0.43750,0.000000,0.819149,0.166667,0.666667,0.631579,0.200000,0.245200,0.600467
18275,0.50000,0.000000,0.691489,0.333333,0.333333,0.631579,0.266667,0.587933,0.767000
18276,0.56250,0.000000,0.574468,0.416667,0.333333,0.578947,0.333333,0.777967,0.871867


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[4.32432432e-01 7.77777778e-02 9.06250000e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.32432432e-01 7.77777778e-02 9.37500000e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.05405405e-01 5.55555556e-02 9.68750000e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [4.32432432e-01 0.00000000e+00 6.97916667e-01 ... 2.66666667e-01
  5.87933333e-01 7.67000000e-01]
 [4.86486486e-01 0.00000000e+00 5.83333333e-01 ... 3.33333333e-01
  7.77966667e-01 8.71866667e-01]
 [5.94594595e-01 0.00000000e+00 4.16666667e-01 ... 4.66666667e-01
  8.75900000e-01 8.55100000e-01]]
(5214, 9)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.432432,0.077778,0.906250,0.000000,0.666667,0.75,0.000000,0.000000,0.000000
31,0.432432,0.077778,0.937500,0.000000,0.333333,0.80,0.066667,0.000000,0.000219
32,0.405405,0.055556,0.968750,0.000000,0.333333,0.75,0.133333,0.000000,0.018681
33,0.432432,0.000000,0.927083,0.083333,0.000000,0.80,0.200000,0.014627,0.257353
34,0.459459,0.000000,0.843750,0.166667,0.000000,0.75,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
18273,0.351351,0.000000,0.864583,0.083333,0.666667,0.55,0.133333,0.002233,0.243400
18274,0.378378,0.000000,0.822917,0.166667,0.666667,0.60,0.200000,0.245200,0.600467
18275,0.432432,0.000000,0.697917,0.333333,0.333333,0.60,0.266667,0.587933,0.767000
18276,0.486486,0.000000,0.583333,0.416667,0.333333,0.55,0.333333,0.777967,0.871867


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.        ]
 [0.00983333]
 [0.31593333]]
(3649, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
12958,0.000000
12967,0.000000
12968,0.000000
12969,0.009833


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[6.50666667e-01]
 [6.78866667e-01]
 [6.56666667e-01]
 [6.34033333e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.17000000e-02]
 [4.02500000e-01]
 [6.29733333e-01]
 [6.02466667e-01]
 [3.92200000e-01]
 [4.24333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.01333333e-02]
 [3.76966667e-01]
 [7.10300000e-01]
 [7.51233333e-01]
 [7.85666667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.57333333e-02]
 [4.93300000e-01]
 [8.19466667e-01]
 [8.34700000e-01]
 [8.20833333e-01]
 [7.17400000e-01]
 [8.18666667e-01]
 [6.90233333e-01]
 [4.11033333e-01]
 [4.34666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.32000000e-02]
 [5.93533333e-01]
 [6.08633333e-01]
 [8.79666667e-01]
 [6.04733333e-01]
 [1.12600000e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.32066667e-01]
 [2.87266667e-01]
 [8.49266667e-01]
 [8.96300000e-01]
 [0.00000000e+00]
 [0.000000

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12971,0.650667
12972,0.678867
12973,0.656667
12974,0.634033
12981,0.000000
...,...
16383,0.804800
16387,0.346933
16388,0.203733
16389,0.014900


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[2.14000000e-02]
 [2.26800000e-01]
 [3.50033333e-01]
 [4.38233333e-01]
 [5.17866667e-01]
 [2.55666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.79666667e-02]
 [2.78633333e-01]
 [4.04633333e-01]
 [3.96600000e-01]
 [3.95833333e-01]
 [3.86166667e-01]
 [5.20300000e-01]
 [4.55600000e-01]
 [3.93566667e-01]
 [2.66166667e-01]
 [2.36933333e-01]
 [2.46166667e-01]
 [2.48333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.20333333e-02]
 [2.95566667e-01]
 [4.52433333e-01]
 [5.00800000e-01]
 [4.54266667e-01]
 [2.79633333e-01]
 [2.82066667e-01]
 [2.80133333e-01]
 [2.70033333e-01]
 [4.37733333e-01]
 [2.42166667e-01]
 [2.97333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.84000000e-02]
 [2.61400000e-01]
 [5.28433333e-01]
 [5.05033333e-01]
 [3.92000000e-01]
 [4.46200000e-01]
 [8.93633333e-01]
 [7.99433333e-01]
 [3.90766667e-01]
 [4.55000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.93000000e-02]
 [3.93100000e-01]
 [7.95400000e-01]
 [9.24766667e-01]
 [0.00000000e+00]
 [4.36666667e-02]
 [4.353333

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
16400,0.021400
16401,0.226800
16402,0.350033
16403,0.438233
16404,0.517867
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.77796667]
 [0.87743333]
 [0.85326667]]
(5214, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


## Preparación para Redes Neuronales

In [30]:
import numpy as np

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [31]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [32]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (3601, 48, 9), y_train: (3601, 1)
X_val: (734, 48, 9), y_val: (734, 1)
X_test: (735, 48, 9), y_test: (735, 1)


## Optuna

In [33]:
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [34]:
# import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50, n_jobs=-1)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-11 21:30:02,773] A new study created in memory with name: no-name-c7cc6fa1-4d78-405b-a08f-62bb04bf7af7
[I 2025-03-11 21:30:05,041] Trial 1 finished with value: 0.014343138831026028 and parameters: {'num_leaves': 826, 'subsample': 0.4113629776985105, 'colsample_bytree': 0.12421653820549444, 'min_data_in_leaf': 55}. Best is trial 1 with value: 0.014343138831026028.
[I 2025-03-11 21:30:05,217] Trial 7 finished with value: 0.00976090659763726 and parameters: {'num_leaves': 397, 'subsample': 0.6390782077948541, 'colsample_bytree': 0.45934163887949353, 'min_data_in_leaf': 94}. Best is trial 7 with value: 0.00976090659763726.
[I 2025-03-11 21:30:05,924] Trial 6 finished with value: 0.009761726564227113 and parameters: {'num_leaves': 729, 'subsample': 0.43301340471536287, 'colsample_bytree': 0.7224846095452567, 'min_data_in_leaf': 99}. Best is trial 7 with value: 0.00976090659763726.
[I 2025-03-11 21:30:07,140] Trial 5 finished with value: 0.010985165612611848 and parameters: {'num_

Mejores hiperparámetros: {'num_leaves': 708, 'subsample': 0.3584326593589673, 'colsample_bytree': 0.4904780669222326, 'min_data_in_leaf': 27}


### Random Forest

In [35]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-11 21:30:26,602] A new study created in memory with name: no-name-b624b924-029c-4e3e-8cf3-7fa9bd36de9f
[I 2025-03-11 21:30:45,890] Trial 5 finished with value: 0.010791807895324695 and parameters: {'n_estimators': 250, 'max_depth': 50, 'min_samples_split': 8, 'min_samples_leaf': 7, 'bootstrap': True}. Best is trial 5 with value: 0.010791807895324695.
[I 2025-03-11 21:30:47,152] Trial 4 finished with value: 0.01105918170256387 and parameters: {'n_estimators': 250, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 3, 'bootstrap': True}. Best is trial 5 with value: 0.010791807895324695.
[I 2025-03-11 21:30:47,771] Trial 3 finished with value: 0.017960573324511436 and parameters: {'n_estimators': 150, 'max_depth': 45, 'min_samples_split': 7, 'min_samples_leaf': 1, 'bootstrap': False}. Best is trial 5 with value: 0.010791807895324695.
[I 2025-03-11 21:30:49,336] Trial 0 finished with value: 0.01084451583013472 and parameters: {'n_estimators': 300, 'max_depth': 25, 'min

Mejores hiperparámetros: {'n_estimators': 450, 'max_depth': 40, 'min_samples_split': 5, 'min_samples_leaf': 6, 'bootstrap': True}


### CTNET

In [36]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    
    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [34]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [38]:
# Definir la función objetivo para Optuna
def objective(trial):
    try:
        # Hiperparámetros a optimizar
        head_size = trial.suggest_int("head_size", 2, 8)
        num_heads = trial.suggest_int("num_heads", 2, 8)
        ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
        num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
        mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                     trial.suggest_int("mlp_units_2", 32, 256, step=32)]
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
        learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

        # Construir el modelo
        model = build_model(
            input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
            head_size=head_size,
            num_heads=num_heads,
            ff_dim=ff_dim,
            num_transformer_blocks=num_transformer_blocks,
            mlp_units=mlp_units,
            dropout=dropout,
            mlp_dropout=mlp_dropout
        )

        # Compilar el modelo
        model.compile(
            loss=tf.keras.losses.MeanSquaredError(),
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            metrics=[tf.keras.metrics.RootMeanSquaredError()]
        )

        # Definir Early Stopping para evitar sobreajuste y consumo innecesario de GPU
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=15,  # 🔹 Detener si no mejora en 15 epochs
            restore_best_weights=True,
            verbose=1
        )

        # Entrenar el modelo con GPU
        history = model.fit(
            X_train_windowed, y_train_windowed,
            epochs=100,
            batch_size=batch_size,
            validation_data=(X_val_windowed, y_val_windowed),
            callbacks=[early_stop],
            verbose=0
        )

        # Obtener la mejor pérdida de validación
        val_loss = min(history.history["val_loss"])
        return val_loss  # Minimizar el error MSE

    except Exception as e:
        print(f"🚨 Error en Optuna: {e}")
        return np.inf  # Si falla, devolver un valor alto para que Optuna lo ignore

# Crear el estudio de optimización con paralelización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50, n_jobs=-1)  # 🔥 Usa 4 hilos para evitar saturación de VRAM

# Imprimir los mejores hiperparámetros encontrados
print("✅ Mejores hiperparámetros:", CTNET_study.best_params)

[I 2025-03-11 21:33:43,908] A new study created in memory with name: no-name-2b26bac5-3084-4f1b-99b8-a57253ec5a4a


Epoch 23: early stopping
Restoring model weights from the end of the best epoch: 8.


[I 2025-03-11 21:38:26,765] Trial 5 finished with value: 0.13148528337478638 and parameters: {'head_size': 5, 'num_heads': 5, 'ff_dim': 112, 'num_transformer_blocks': 1, 'mlp_units_1': 256, 'mlp_units_2': 256, 'dropout': 0.1464609051576174, 'mlp_dropout': 0.31120598758948154, 'learning_rate': 0.002300860263595248, 'batch_size': 256}. Best is trial 5 with value: 0.13148528337478638.


Epoch 47: early stopping
Restoring model weights from the end of the best epoch: 32.


[I 2025-03-11 21:58:27,589] Trial 0 finished with value: 0.0580473318696022 and parameters: {'head_size': 4, 'num_heads': 3, 'ff_dim': 112, 'num_transformer_blocks': 1, 'mlp_units_1': 448, 'mlp_units_2': 224, 'dropout': 0.28870658692025264, 'mlp_dropout': 0.4301413362929669, 'learning_rate': 0.000753151615081156, 'batch_size': 128}. Best is trial 0 with value: 0.0580473318696022.


...

Epoch 64: early stopping
Restoring model weights from the end of the best epoch: 49.


[I 2025-03-12 10:08:37,297] Trial 45 finished with value: 0.055296391248703 and parameters: {'head_size': 5, 'num_heads': 5, 'ff_dim': 80, 'num_transformer_blocks': 3, 'mlp_units_1': 192, 'mlp_units_2': 224, 'dropout': 0.46408960121891457, 'mlp_dropout': 0.12571740680220642, 'learning_rate': 0.0017791790013173883, 'batch_size': 256}. Best is trial 40 with value: 0.05336686596274376.


Epoch 87: early stopping
Restoring model weights from the end of the best epoch: 72.


[I 2025-03-12 10:26:38,096] Trial 48 finished with value: 0.056557804346084595 and parameters: {'head_size': 5, 'num_heads': 5, 'ff_dim': 128, 'num_transformer_blocks': 3, 'mlp_units_1': 192, 'mlp_units_2': 64, 'dropout': 0.4624707266085965, 'mlp_dropout': 0.1246582197948072, 'learning_rate': 0.0009942796951539427, 'batch_size': 256}. Best is trial 40 with value: 0.05336686596274376.


✅ Mejores hiperparámetros: {'head_size': 4, 'num_heads': 5, 'ff_dim': 128, 'num_transformer_blocks': 3, 'mlp_units_1': 128, 'mlp_units_2': 64, 'dropout': 0.10435760102849667, 'mlp_dropout': 0.27390486046392415, 'learning_rate': 0.001055563359247843, 'batch_size': 256}


### Forescasting

In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())  
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-12 10:26:38,337] A new study created in memory with name: no-name-a5035ae3-1b11-4241-9d35-28b61b8165fe


[I 2025-03-12 10:37:44,462] Trial 6 finished with value: 0.276267409324646 and parameters: {'filters': 64, 'kernel_size': 5, 'lstm_units_1': 256, 'lstm_units_2': 64, 'lstm_units_3': 64, 'dropout_lstm': 0.13685924874283406, 'dropout_dense': 0.17855295556451162, 'learning_rate': 0.0059779984678818895, 'batch_size': 128}. Best is trial 6 with value: 0.276267409324646.
[I 2025-03-12 10:43:32,753] Trial 8 finished with value: 0.2788870930671692 and parameters: {'filters': 128, 'kernel_size': 3, 'lstm_units_1': 128, 'lstm_units_2': 128, 'lstm_units_3': 32, 'dropout_lstm': 0.4449970723598019, 'dropout_dense': 0.4965017863116885, 'learning_rate': 0.00017166429477555824, 'batch_size': 512}. Best is trial 6 with value: 0.276267409324646.
[I 2025-03-12 10:51:11,328] Trial 9 finished with value: 0.27658557891845703 and parameters: {'filters': 128, 'kernel_size': 5, 'lstm_units_1': 64, 'lstm_units_2': 32, 'lstm_units_3': 16, 'dropout_lstm': 0.15750743458769936, 'dropout_dense': 0.44497142075442353,

Mejores hiperparámetros: {'filters': 128, 'kernel_size': 4, 'lstm_units_1': 64, 'lstm_units_2': 64, 'lstm_units_3': 32, 'dropout_lstm': 0.12850289949645297, 'dropout_dense': 0.3736924023288869, 'learning_rate': 0.0005232498319662458, 'batch_size': 512}


### Photovoltaic

In [35]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_j
                            obs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-13 10:17:18,108] A new study created in memory with name: no-name-560085e2-0501-46a3-88e2-468c9f1bd050


Epoch 40: early stopping
Restoring model weights from the end of the best epoch: 30.


[I 2025-03-13 10:46:36,165] Trial 0 finished with value: 0.06718964129686356 and parameters: {'filters_1': 32, 'filters_2': 256, 'kernel_size': 3, 'gru_units': 64, 'num_heads': 4, 'dropout_rate': 0.25017037807866194, 'dense_units_1': 64, 'dense_units_2': 64, 'learning_rate': 0.0010720966270651466, 'batch_size': 512}. Best is trial 0 with value: 0.06718964129686356.


Epoch 64: early stopping
Restoring model weights from the end of the best epoch: 54.


[I 2025-03-13 11:06:14,498] Trial 7 finished with value: 0.06049086153507233 and parameters: {'filters_1': 32, 'filters_2': 128, 'kernel_size': 3, 'gru_units': 64, 'num_heads': 8, 'dropout_rate': 0.20741041595352808, 'dense_units_1': 32, 'dense_units_2': 32, 'learning_rate': 0.0014519496005881705, 'batch_size': 512}. Best is trial 7 with value: 0.06049086153507233.


...

Epoch 34: early stopping
Restoring model weights from the end of the best epoch: 24.


[I 2025-03-13 20:30:22,636] Trial 46 finished with value: 0.057010967284440994 and parameters: {'filters_1': 64, 'filters_2': 128, 'kernel_size': 4, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.2907935947667643, 'dense_units_1': 32, 'dense_units_2': 16, 'learning_rate': 0.0021359368079330826, 'batch_size': 128}. Best is trial 21 with value: 0.05316260829567909.


Epoch 44: early stopping
Restoring model weights from the end of the best epoch: 34.


[I 2025-03-13 20:36:51,908] Trial 45 finished with value: 0.05312512069940567 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 4, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.28918099674260234, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.0020715275009286437, 'batch_size': 128}. Best is trial 45 with value: 0.05312512069940567.


Mejores hiperparámetros: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 4, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.28918099674260234, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.0020715275009286437, 'batch_size': 128}
